# Job Task: Train All Models

> **This notebook runs as a Databricks Job task.**

Task 2 of the retrain pipeline:
- Train Logistic Regression, Random Forest, and Gradient Boosted Trees
- Log all runs to MLflow under a shared experiment
- Pass the best run_id to downstream tasks via task values

In [0]:
dbutils.widgets.text("catalog", "workshop", "Catalog")
dbutils.widgets.text("schema", "00_shared", "Schema")
dbutils.widgets.text("experiment_name", "experiment", "Experiment Name")
dbutils.widgets.text("config_path", "config.yml", "Config Path")
dbutils.widgets.text("source_table", "", "Source Table (leave blank for default)")

catalog = dbutils.widgets.get("catalog")
schema  = dbutils.widgets.get("schema")
source_table_override = dbutils.widgets.get("source_table")

In [0]:
%pip install /Volumes/{catalog}/00_shared/wheels/churn_model-0.1.0-py3-none-any.whl --quiet

In [0]:
dbutils.library.restartPython()

In [0]:
import yaml

catalog = dbutils.widgets.get("catalog")
schema  = dbutils.widgets.get("schema")
experiment_name = dbutils.widgets.get("experiment_name")
config_path  = dbutils.widgets.get("config_path")
source_table_override = dbutils.widgets.get("source_table")

with open(config_path) as f:
    config = yaml.safe_load(f)

source_table = source_table_override if source_table_override else f"{catalog}.`00_shared`.telco_churn"
print(f"Source table: {source_table}")

from churn_model.train import run_all_models
results = run_all_models(
    catalog=catalog,
    schema=schema,
    config=config,
    experiment_name=experiment_name,
    source_table=source_table,
)

from churn_model.evaluate import get_best_run
best_run_id, best_model_type, best_metrics = get_best_run(
    experiment_name=experiment_name,
    metric="test_f1",
)

print(f"Best model  : {best_model_type}")
print(f"Best run ID : {best_run_id}")
print(f"F1          : {best_metrics['test_f1']:.4f}")

# Pass to downstream tasks
dbutils.jobs.taskValues.set("best_run_id", best_run_id)
dbutils.jobs.taskValues.set("best_model_type", best_model_type)
dbutils.jobs.taskValues.set("best_f1", str(best_metrics["test_f1"]))